In [2]:
import pandas as pd

hemo_df = pd.read_csv(
    "../data/processed_data&code/hemolysis_dataset.csv"
)

print(hemo_df.shape)
print(
    hemo_df["hemo_label"].value_counts()
)

(1509, 8)
hemo_label
0.0    942
1.0    567
Name: count, dtype: int64


In [3]:
print(hemo_df.columns.tolist())

['DRAMP_ID', 'Sequence', 'Sequence_Length', 'Activity', 'Hemolytic_activity', 'Target_Organism', 'hemo_label', 'max_hemo_percent']


In [4]:
import torch
import esm

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()

model = model.to(device)
model.eval()

batch_converter = alphabet.get_batch_converter()

print("ESM Loaded")

ESM Loaded


In [5]:
import numpy as np
from tqdm import tqdm

hemo_sequences = hemo_df["Sequence"].tolist()

batch_size = 64

hemo_embeddings = []

for i in tqdm(range(0, len(hemo_sequences), batch_size)):

    batch_seqs = hemo_sequences[i:i+batch_size]

    batch = [
        (str(j), seq)
        for j, seq in enumerate(batch_seqs)
    ]

    _, _, tokens = batch_converter(batch)

    tokens = tokens.to(device)

    with torch.no_grad():

        results = model(
            tokens,
            repr_layers=[33],
            return_contacts=False
        )

    reps = results["representations"][33]

    for j, seq in enumerate(batch_seqs):

        seq_len = len(seq)

        emb = (
            reps[j, 1:seq_len+1]
            .mean(0)
            .cpu()
            .numpy()
        )

        hemo_embeddings.append(emb)

hemo_embeddings = np.array(
    hemo_embeddings
)

print(hemo_embeddings.shape)

100%|███████████████████████████████████████████████████████████████████████████████████| 24/24 [00:04<00:00,  5.49it/s]

(1509, 1280)


In [6]:
import numpy as np

np.save(
    "../embeddings/hemo_embeddings.npy",
    hemo_embeddings
)

hemo_df.to_csv(
    "../embeddings/hemo_metadata.csv",
    index=False
)

print("Saved")

Saved


In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

X = np.load(
    "../embeddings/hemo_embeddings.npy"
)

meta = pd.read_csv(
    "../embeddings/hemo_metadata.csv"
)

y = meta["hemo_label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

print("Train Labels")
print(pd.Series(y_train).value_counts())

print("Test Labels")
print(pd.Series(y_test).value_counts())

(1207, 1280)
(302, 1280)
Train Labels
0.0    753
1.0    454
Name: count, dtype: int64
Test Labels
0.0    189
1.0    113
Name: count, dtype: int64


In [8]:
from xgboost import XGBClassifier

ratio = 753 / 454

hemo_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=ratio,
    tree_method="hist",
    random_state=42
)

hemo_model.fit(
    X_train,
    y_train
)

print("Hemolysis Training Complete")

Hemolysis Training Complete


In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

y_pred = hemo_model.predict(X_test)

y_prob = hemo_model.predict_proba(X_test)[:,1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(
    classification_report(
        y_test,
        y_pred,
        digits=4
    )
)

Accuracy : 0.7947019867549668
Precision: 0.7741935483870968
Recall   : 0.6371681415929203
F1 Score : 0.6990291262135923
ROC-AUC  : 0.8769021866366999

Confusion Matrix
[[168  21]
 [ 41  72]]

Classification Report
              precision    recall  f1-score   support

         0.0     0.8038    0.8889    0.8442       189
         1.0     0.7742    0.6372    0.6990       113

    accuracy                         0.7947       302
   macro avg     0.7890    0.7630    0.7716       302
weighted avg     0.7927    0.7947    0.7899       302



In [10]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score
)

rf_prob = rf.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, rf_pred))
print("F1:", f1_score(y_test, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf_prob))

Accuracy: 0.7847682119205298
F1: 0.6632124352331606
ROC-AUC: 0.8702533127311889


In [11]:
import joblib

joblib.dump(
    hemo_model,
    "hemolysis_xgb_classifier.pkl"
)

print("Saved")

Saved


In [12]:
loaded_model = joblib.load(
    "hemolysis_xgb_classifier.pkl"
)

print(type(loaded_model))

<class 'xgboost.sklearn.XGBClassifier'>
